In [1]:
import numpy as np
# import seaborn as sns
import pandas as pd
import os.path

import matplotlib.pyplot as plt

import tifffile 
import czifile

from skimage import transform
from scipy import ndimage

import random 
import math

In [2]:
patch_size = 32
ctrl_y_str = 'ctrl_ch1_major'
major_ch = 1

start_ind = 0
end_ind = 51

In [3]:
image_folder = '/mnt/d/lding/FA/data/FA_ML_Annabel_20250217/031125data/Control'
cell_mask_folder = '/mnt/d/lding/FA/analysis_results/FA_ML_Annabel_20250217/031125/'+ctrl_y_str+'/code_org_20250820_seg/mask'

front_mask_dir = '/mnt/d/lding/FA/data/FA_ML_Annabel_20250217/031125data/frontmasks'

In [4]:
half_ps = int(patch_size/2)
half_half_ps = int(patch_size/4)
double_ps = patch_size*2
double_double_ps = patch_size*4

In [5]:
def rotate_coor(x_i,y_i,x_c,y_c,rotate_angle):
 
    rotate_angle = rotate_angle*np.pi/180
 
    x_o = (x_i-x_c)*math.cos(rotate_angle) - (2*y_c-y_i-y_c)*math.sin(rotate_angle) +x_c
    y_o = -(x_i-x_c)*math.sin(rotate_angle) - (2*y_c-y_i-y_c)*math.cos(rotate_angle) +(2*y_c-y_c)

    return([x_o,y_o])

In [6]:
from datetime import datetime
now = datetime.now()

time_str = now.strftime("%Y%m%d-%H%M")

In [ ]:

mask_ratio = 0.65
movie_partitioned_data_dir = '/mnt/d/lding/FA/analysis_results/FA_ML_Annabel_20250217/031125data/'\
    +ctrl_y_str+'/tiff_HH_gridonly_patches'+str(patch_size)+'_65p_'+time_str
movie_plot_dir = '/mnt/d/lding/FA/analysis_results/FA_ML_Annabel_20250217/031125/' \
    +ctrl_y_str+'/plot_HH_gridonly_patches_ps'+str(patch_size)+'_65p_'+time_str
os.makedirs(movie_partitioned_data_dir,exist_ok=True)
os.makedirs(movie_plot_dir,exist_ok=True)


In [8]:
movie_partitioned_data_dir

'/mnt/d/lding/FA/analysis_results/FA_ML_Annabel_20250217/031125data/ctrl_ch1_major/tiff_HH_localmax_patches32_65p_20250907-0013'

In [9]:
data_prep_record = pd.DataFrame(columns={'image_folder','filename','filenameID','x_c','y_c','rand_angle','rand_tx','rand_ty',
                            'x_corner1','x_corner2','x_corner3','x_corner4','y_corner1','y_corner2','y_corner3','y_corner4',
                            'movie_partitioned_data_dir','crop_img_filename','movie_plot_dir','plot_filename'})

In [10]:
from skimage.feature import peak_local_max



In [ ]:
filenames = [x for x in os.listdir(image_folder) if os.path.isfile(os.path.join(image_folder, x)) and ('.czi' in x)]

debug_flag = 0

for filenameID in range(start_ind, min(end_ind,len(filenames))):
# for filenameID in range(0,1):
    if debug_flag ==1:
                break
    filename = filenames[filenameID]
    train_img = czifile.imread(os.path.join(image_folder, filename)).squeeze()[major_ch,:,:].astype(float)/255/255
    train_seg = tifffile.imread(os.path.join(cell_mask_folder, "cell_mask_"+filename+".tif")).squeeze().astype(float)

    if os.path.isfile(os.path.join(front_mask_dir,'frontmask-'+filename[:-4]+'.tif')):
        front_mask = tifffile.imread(os.path.join(front_mask_dir,'frontmask-'+filename[:-4]+'.tif'))  
        train_seg = (train_seg *  front_mask)>0

    fig_accu, ax_accu = plt.subplots(1,2, figsize=(15.6,7.8), dpi=256, facecolor='w', edgecolor='k')
    ax_accu[0].imshow(train_img, cmap=plt.cm.gray,vmax=1,vmin=0)
    ax_accu[1].imshow(train_seg, cmap=plt.cm.gray,vmax=1,vmin=0)    
    
    x_num = int(np.floor(train_img.shape[1]/patch_size))
    y_num = int(np.floor(train_img.shape[0]/patch_size))
    
    x_0 = int((train_img.shape[1] - x_num*patch_size)/2+ (0.5)*patch_size)
    y_0 = int((train_img.shape[0] - y_num*patch_size)/2+ (0.5)*patch_size)
    
            
    for x_i in range(x_num):
        if debug_flag ==1:
                break
        for y_i in range(x_num):

            if debug_flag ==1:
                break

            y_c = int(y_0+(y_i-0.5)*patch_size)
            x_c = int(x_0+(x_i-0.5)*patch_size)

            y_left = y_c - double_ps
            x_left = x_c - double_ps

            y_right = y_c + double_ps
            x_right = x_c + double_ps


            if y_left < 0 or x_left < 0 or y_right >= train_img.shape[0] or x_right >= train_img.shape[1]:
                continue

            patch_img = train_img[y_left:y_right,x_left:x_right]
            patch_seg = train_seg[y_left:y_right,x_left:x_right]
            
            if((patch_seg.mean())<mask_ratio/16):
                continue
            
            # coords = peak_local_max(patch_img, min_distance=3, threshold_abs=0.01)
            
            # coords = coords - double_ps
            # mask = (np.abs(coords) < half_half_ps).all(axis=1)
            # filtered_coords = coords[mask]
            
            # if len(filtered_coords) > 0:
            #     # Convert back to absolute indices in patch_img
            #     abs_coords = filtered_coords + double_ps
            #     # Get values at those coordinates
            #     values = patch_img[abs_coords[:, 0], abs_coords[:, 1]]
            #     # Index of max
            #     max_idx = np.argmax(values)
            #     best_coord = filtered_coords[max_idx]
            #     # 1 is x, 0 is y
            #     # translate  with -
            #     rand_tx = -best_coord[1]
            #     rand_ty = -best_coord[0]            
            # else:
            #     best_coord = None  # no valid peaks found
            #     continue

            rand_tx = 0
            rand_ty = 0

            # crop around the local maximum first
            cx_left_1 = patch_size-rand_tx
            cx_right_1 = double_ps+patch_size-rand_tx
            cy_up_1 = patch_size-rand_ty
            cy_down_1 = double_ps+patch_size-rand_ty

            big_crop_patch_img = patch_img[cy_up_1:cy_down_1,
                                       cx_left_1:cx_right_1]
            big_crop_patch_seg = patch_seg[cy_up_1:cy_down_1,
                                       cx_left_1:cx_right_1]
            
            first_crop_x = np.array([cx_left_1, cx_left_1, cx_right_1, cx_right_1, cx_left_1]) + x_left
            first_crop_y = np.array([cy_up_1, cy_down_1, cy_down_1, cy_up_1, cy_up_1]) + y_left            


            if((big_crop_patch_seg.mean())<mask_ratio/4):
                continue

            rand_angle = random.random() * 0
            rotated_patch_img = transform.rotate(big_crop_patch_img, rand_angle, resize=False, center=None, order=None, mode='constant', cval=0, clip=True)
            rotated_patch_seg = transform.rotate(big_crop_patch_seg, rand_angle, resize=False, center=None, order=None, mode='constant', cval=0, clip=True)
          
            cx_left_2 = half_ps
            cx_right_2 = patch_size+half_ps
            cy_up_2 = half_ps
            cy_down_2 = patch_size+half_ps

            crop_patch_img = rotated_patch_img[cy_up_2:cy_down_2,
                                       cx_left_2:cx_right_2]
            crop_patch_seg = rotated_patch_seg[cy_up_2:cy_down_2,
                                       cx_left_2:cx_right_2]

            if((crop_patch_seg.mean())<mask_ratio):
                continue

            crop_img_filename = 'c'+str(filenameID).zfill(4)+'x'+str(x_c).zfill(4)+'y'+str(y_c).zfill(4)+'ps'+str(patch_size)+'.tif'
            
            tifffile.imwrite(
                os.path.join(movie_partitioned_data_dir,crop_img_filename),
                crop_patch_img.astype(np.float32),
                imagej=True,              # Write ImageJ metadata block
                metadata={'axes': 'YX'}   # Or 'TYX', 'ZYX', etc. depending on shape
            )

            X_TransRotCrop = np.array([cx_left_2, cx_left_2, cx_right_2, cx_right_2, cx_left_2])
            Y_TransRotCrop = np.array([cy_up_2, cy_down_2, cy_down_2, cy_up_2, cy_up_2])

            [X_invrotate_patch, Y_invrotate_patch] = rotate_coor(X_TransRotCrop,Y_TransRotCrop,patch_size,patch_size,-rand_angle)

            X_bigger_patch = X_invrotate_patch + x_left + cx_left_1
            Y_bigger_patch = Y_invrotate_patch + y_left + cy_up_1


            # fig, ax = plt.subplots(2,4, figsize=(15.6,7.8), dpi=256, facecolor='w', edgecolor='k')
            # ax[0,0].imshow(train_img, cmap=plt.cm.gray,vmax=1,vmin=0)
            # ax[0,0].plot([x_left, x_left, x_right, x_right, x_left],[y_left,y_right, y_right, y_left,y_left],color='r')
            # ax[0,0].plot(X_bigger_patch, Y_bigger_patch,color='magenta')

            # ax[0,1].imshow(train_seg, cmap=plt.cm.gray,vmax=1,vmin=0)
            # ax[0,1].plot([x_left, x_left, x_right, x_right, x_left],[y_left,y_right, y_right, y_left,y_left],color='r')
            # ax[0,1].plot(first_crop_x,first_crop_y,color='g')
            # ax[0,1].plot(X_bigger_patch, Y_bigger_patch,color='magenta')


            # ax[0,2].imshow(patch_img, cmap=plt.cm.gray,vmax=1,vmin=0)
            # ax[0,2].plot([cx_left_1,cx_left_1,cx_right_1,cx_right_1,cx_left_1],
            #              [cy_up_1,cy_down_1,cy_down_1,cy_up_1,cy_up_1], color='green')
            # ax[0,2].plot(X_invrotate_patch +cx_left_1,Y_invrotate_patch+cy_up_1,color='magenta')

            # ax[0,3].imshow(patch_seg, cmap=plt.cm.gray,vmax=1,vmin=0)
            # ax[0,3].plot([cx_left_1,cx_left_1,cx_right_1,cx_right_1,cx_left_1],
            #              [cy_up_1,cy_down_1,cy_down_1,cy_up_1,cy_up_1],color='green')
            # ax[0,3].plot(X_invrotate_patch +cx_left_1,Y_invrotate_patch+cy_up_1,color='magenta')

            # ax[1,0].imshow(rotated_patch_img, cmap=plt.cm.gray,vmax=1,vmin=0)
            # ax[1,0].plot([half_ps, half_ps, patch_size+half_ps, patch_size+half_ps, half_ps],
            #              [half_ps,patch_size+half_ps, patch_size+half_ps, half_ps,half_ps],color='magenta')
          
            # ax[1,1].imshow(rotated_patch_seg, cmap=plt.cm.gray,vmax=1,vmin=0)
            # ax[1,1].plot([half_ps, half_ps, patch_size+half_ps, patch_size+half_ps, half_ps],
            #              [half_ps,patch_size+half_ps, patch_size+half_ps, half_ps,half_ps],color='magenta')

            # ax[1,2].imshow(crop_patch_img, cmap=plt.cm.gray,vmax=1,vmin=0)
            # ax[1,3].imshow(crop_patch_seg, cmap=plt.cm.gray,vmax=1,vmin=0)
            
            plot_filename = 'plot_grid_t'+str(filenameID).zfill(4)+'_xc'+str(x_c)+'_yc'+str(y_c)+ '.png'
            # # debug_flag = 1
            # fig.savefig(os.path.join(movie_plot_dir,plot_filename),bbox_inch='tight')
            # # break
            # plt.close(fig) 
            
            ax_accu[0].plot(X_bigger_patch, Y_bigger_patch,color='green')
            ax_accu[1].plot(X_bigger_patch, Y_bigger_patch,color='green')
            
            s = pd.Series([image_folder, filename, filenameID, x_c,y_c,rand_angle,rand_tx,rand_ty,
                X_bigger_patch[0],X_bigger_patch[1],X_bigger_patch[2],X_bigger_patch[3],
                Y_bigger_patch[0],Y_bigger_patch[1],Y_bigger_patch[2],Y_bigger_patch[3],
                movie_partitioned_data_dir,crop_img_filename,movie_plot_dir,plot_filename],
                index=['image_folder','filename','filenameID','x_c','y_c','rand_angle','rand_tx','rand_ty',
                                        'x_corner1','x_corner2','x_corner3','x_corner4','y_corner1','y_corner2','y_corner3','y_corner4',
                                        'movie_partitioned_data_dir','crop_img_filename','movie_plot_dir','plot_filename'])
            
            data_prep_record = data_prep_record.append(s,ignore_index=True)

    data_prep_record.to_csv(os.path.join(movie_plot_dir,'data_prep_record_'+str(filenameID)+'_t'+str(filenameID)+'.csv'))

    fig_accu.savefig(os.path.join(movie_plot_dir,'grid_t'+str(filenameID).zfill(4)+'.png'))   
    plt.close(fig_accu)   
            

IndexError: list index out of range